In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score

benchmark = pd.read_csv("data/benchmark.csv")  # default: sample benchmark table from scripts/generate_sample_data.py; replace with your own benchmark CSV
pb_data = pd.read_csv("data/processed_epb.csv")  # default: sample processed TorchPB table from scripts/generate_sample_data.py; replace with your own processed CSV

In [ ]:
benchmark.info()
pb_data.info()

In [ ]:
display(benchmark.sample(5))
display(pb_data.sample(5))

In [ ]:
benchmark["Atom Number"] = benchmark["Atom Number"].astype(str)
benchmark = benchmark[["Split", "Mol Name", "Atom Number", "Grid Size", "Solver Name", "Tolerance", "EPB Value", "Elapsed Time", "CUDA Used", "Succeeded"]]

pb_data["Atom Number"] = pb_data["Atom Number"].astype(str)
pb_data = pb_data[["Split", "Mol Name", "Atom Number", "Grid Size", "Solver Name", "Tolerance", "EPB Value", "Elapsed Time", "CUDA Used", "Succeeded"]]

In [ ]:
df = pd.merge(
    benchmark,
    pb_data,
    on = ["Split", "Atom Number", "Mol Name", "Grid Size", "CUDA Used", "Tolerance"],
    how = "inner",
    suffixes = ("_Bench", "_Data")
).rename(columns = {"EPB Value_Bench" : "EPB_Bench", "EPB Value_Data" : "EPB_Data", "Solver Name_Data" : "Solver Name"}).drop("Solver Name_Bench", axis = 1)
df = df[["Split", "Mol Name", "Atom Number", "Solver Name", "Grid Size", "Tolerance", "CUDA Used", "EPB_Bench", "EPB_Data", "Succeeded_Bench", "Succeeded_Data", "Elapsed Time_Bench", "Elapsed Time_Data"]]

In [ ]:
df

In [ ]:
print(df[["EPB_Bench", "EPB_Data"]].dtypes)

In [ ]:
df["EPB_Bench"] = pd.to_numeric(df["EPB_Bench"].astype(str).str.replace(",", ""), errors="coerce")
df["EPB_Data"] = pd.to_numeric(df["EPB_Data"].astype(str).str.replace(",", ""), errors="coerce")

In [ ]:
df["EPB_abs_diff"] = (df["EPB_Bench"] - df["EPB_Data"]).abs()

In [ ]:
df = df.sort_values(by=["Split", "Mol Name", "CUDA Used"], ascending=True)

In [ ]:
df["Tolerance"].value_counts()

In [ ]:
df_cpu = df[df["CUDA Used"] == 0]

In [ ]:
df_gpu = df[df["CUDA Used"] == 1]

In [ ]:
mask = np.isfinite(df_cpu["EPB_Bench"]) & np.isfinite(df_cpu["EPB_Data"])
df_plot = df_cpu.loc[mask].copy()

splits = sorted(df_plot["Split"].unique())
colors = sns.color_palette("deep", len(splits))

g = sns.jointplot(
    data=df_plot,
    x="EPB_Bench", y="EPB_Data",
    hue="Split",
    kind="scatter",
    palette="deep", 
    marginal_ticks=False,
    height=6, 
    alpha = 0.8
)
g.set_axis_labels("Classical CG (CPU)", "Torchpb CG (CPU)")

g.ax_joint.axline((0, 0), (1, 1), linestyle="--", color="gray", linewidth=1)

split_r2 = df_plot.groupby("Split").apply(lambda d: r2_score(d["EPB_Bench"], d["EPB_Data"])).dropna()

x0, y0, dy = 0.85, 0.2, 0.05
for i, (k, v) in enumerate(split_r2.iloc[::-1].items()):
    idx = splits.index(k)
    g.ax_joint.text(
        x0, y0 + i*dy,
        f"R\u00b2: {v:.4f}",
        transform=g.ax_joint.transAxes,
        ha="right", va="bottom",
        fontsize=16,
        color=colors[idx]
    )

plt.show()

In [ ]:
mask = np.isfinite(df_cpu["EPB_Bench"]) & np.isfinite(df_cpu["EPB_Data"])
df_plot = df_cpu.loc[mask].copy()

splits = sorted(df_plot["Grid Size"].unique())
colors = sns.color_palette("deep", len(splits))

g = sns.jointplot(
    data=df_plot,
    x="EPB_Bench", y="EPB_Data",
    hue="Grid Size",
    kind="scatter",
    palette="deep", 
    marginal_ticks=False,
    height=6
)
g.set_axis_labels("Classical CG (CPU)", "Torchpb CG (CPU)")

g.ax_joint.axline((0, 0), (1, 1), linestyle="--", color="gray", linewidth=1)

split_r2 = df_plot.groupby("Grid Size").apply(lambda d: r2_score(d["EPB_Bench"], d["EPB_Data"])).dropna()

x0, y0, dy = 0.85, 0.2, 0.05
for i, (k, v) in enumerate(split_r2.iloc[::-1].items()):
    idx = splits.index(k)
    g.ax_joint.text(
        x0, y0 + i*dy,
        f"R\u00b2: {v:.4f}",
        transform=g.ax_joint.transAxes,
        ha="right", va="bottom",
        fontsize=16,
        color=colors[idx]
    )

plt.show()

In [ ]:
mask_g = np.isfinite(df_gpu["EPB_Bench"]) & np.isfinite(df_gpu["EPB_Data"])
df_plot_g = df_gpu.loc[mask_g].copy()

splits_g = sorted(df_plot_g["Split"].unique())
colors_g = sns.color_palette("deep", len(splits_g))

g = sns.jointplot(
    data=df_plot_g,
    x="EPB_Bench", y="EPB_Data",
    hue="Split",
    kind="scatter",
    palette="deep",
    marginal_ticks=False,
    height=6
)
g.set_axis_labels("Classical CG (GPU)", "Torchpb CG (GPU)")

g.ax_joint.axline((0, 0), (1, 1), linestyle="--", color="gray", linewidth=1)

split_r2_g = df_plot_g.groupby("Split").apply(lambda d: r2_score(d["EPB_Bench"], d["EPB_Data"])).dropna()


x0, y0, dy = 0.85, 0.2, 0.05
for i, (k, v) in enumerate(split_r2_g.iloc[::-1].items()):
    idx = splits_g.index(k)
    g.ax_joint.text(
        x0, y0 + i*dy,
        f"R\u00b2: {v:.4f}",
        transform=g.ax_joint.transAxes,
        ha="right", va="bottom",
        fontsize=16,
        color=colors_g[idx]
    )

plt.show()

In [ ]:
mask_g = np.isfinite(df_gpu["EPB_Bench"]) & np.isfinite(df_gpu["EPB_Data"])
df_plot_g = df_gpu.loc[mask_g].copy()

splits_g = sorted(df_plot_g["Grid Size"].unique())
colors_g = sns.color_palette("deep", len(splits_g))

g = sns.jointplot(
    data=df_plot_g,
    x="EPB_Bench", y="EPB_Data",
    hue="Grid Size",
    kind="scatter",
    palette="deep",
    marginal_ticks=False,
    height=6
)
g.set_axis_labels("Classical CG (GPU)", "Torchpb CG (GPU)")

g.ax_joint.axline((0, 0), (1, 1), linestyle="--", color="gray", linewidth=1)

split_r2_g = df_plot_g.groupby("Grid Size").apply(lambda d: r2_score(d["EPB_Bench"], d["EPB_Data"])).dropna()


x0, y0, dy = 0.85, 0.2, 0.05
for i, (k, v) in enumerate(split_r2_g.iloc[::-1].items()):
    idx = splits_g.index(k)
    g.ax_joint.text(
        x0, y0 + i*dy,
        f"R\u00b2: {v:.4f}",
        transform=g.ax_joint.transAxes,
        ha="right", va="bottom",
        fontsize=16,
        color=colors_g[idx]
    )

plt.show()

In [ ]:
df_log = df[df["Succeeded_Data"] == 1].copy()
df_log["log_abs_diff"] = np.log10(df_log["EPB_abs_diff"].replace(0, np.nan))

plt.figure(figsize=(10,6))
sns.violinplot(
    data=df_log, 
    x="Grid Size", 
    y="log_abs_diff", 
    hue="Split",
    density_norm="width",
)
plt.ylabel("log(|ΔEPB|)")
plt.title("distribution across Grid Size & Split")
plt.legend(loc="lower left")
plt.show()


In [ ]:
df_log = df[df["Succeeded_Data"] == 1].copy()
df_log["log_abs_diff"] = np.log10(df_log["EPB_abs_diff"].replace(0, np.nan))

plt.figure(figsize=(10,6))
sns.violinplot(
    data=df_log, 
    x="Tolerance", 
    y="log_abs_diff", 
    hue="Split",
)
plt.ylabel("log(|ΔEPB|)")
plt.title("distribution across Tolerance & Split")
plt.legend(loc="upper right")
plt.show()